# 3장 어텐션 메커니즘 구현하기: 따라 치기 노트북

『밑바닥부터 만들면서 배우는 LLM』 3장을 따라 치며 공부하는 노트북입니다. 2장과 같은 방식이에요.

1. 절마다 책을 보며 `✏️` 셀에 코드를 직접 입력하고 실행하세요. 셀은 필요한 만큼 추가하면 됩니다.
2. 주석은 **"이 줄이 왜 필요한지"를 내 말로** 쓰세요.
3. 🔍 **확인 셀**에서 `✅`가 나오면 다음 절로 넘어가세요.

**이번 확인 셀은 숫자까지 책과 비교해요.** 2장에서 확인 셀을 통과했는데도 결과가 책과 달랐던 일을 막기 위해서예요. 그래서 책처럼 `torch.manual_seed(...)`를 **같은 위치에서** 호출해야 같은 숫자가 나와요.

**공부 노트와 연결**: Do it! 정리 노트의 "셀프 어텐션" 부분(Q, K, V, $\text{softmax}(QK^T/\sqrt{d_k})V$)을 이번에 직접 코드로 만들어요.

막히면 [책의 공식 코드](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch03/01_main-chapter-code)를 참고하세요.

## 0. 환경 설정
3장은 PyTorch만 있으면 됩니다. GPU가 없어도 돼요.

In [24]:
import torch
print("torch version:", torch.__version__)


def _close(actual, expected, atol=1e-3):
    """확인 셀용: 책의 출력값과 소수점 셋째 자리까지 같은지"""
    return torch.allclose(torch.as_tensor(actual).detach().float(), torch.tensor(expected), atol=atol)

torch version: 2.14.1+cpu


## 3.1 긴 시퀀스 모델링의 문제점
## 3.2 어텐션 메커니즘으로 데이터 의존성 포착하기
코드 없이 개념을 읽는 절입니다. 읽고 나서 **"RNN은 왜 긴 문장에서 약할까?"** 를 `notes.md`에 한 문장으로 적어 보세요. Do it! 노트의 "RNN은 길어질수록 앞 단어를 잊는다"와 비교해 보세요.

## 3.3 셀프 어텐션으로 입력의 서로 다른 부분에 주의 기울이기
### 3.3.1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션
"Your journey starts with one step" 여섯 단어의 3차원 임베딩 `inputs`를 만들고, 두 번째 단어(journey)를 쿼리로 **어텐션 점수 → 어텐션 가중치 → 컨텍스트 벡터** 순서로 계산합니다.

In [25]:
# ✏️ 책을 보며 직접 입력하세요: inputs 텐서 (6 × 3)
import torch
inputs=torch.tensor([
[0.43, 0.15, 0.89], #Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10],# one
[0.05, 0.80, 0.55]] #step
)
print(inputs)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])


In [26]:
# ✏️ 책을 보며 직접 입력하세요: 두 번째 토큰을 쿼리로 어텐션 점수 attn_scores_2 (내적)
query = inputs[1]
atten_scores_2=torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
  atten_scores_2[i]=torch.dot(x_i,query) #inputs의 모든 x요소들에 대해서 닷연산을 한뒤 저장.
print(atten_scores_2) #어텐션 점수



tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [27]:
# ✏️ 책을 보며 직접 입력하세요: 정규화: 단순 합 나누기, softmax_naive, torch.softmax → attn_weights_2
atten_weights_2_tmp=atten_scores_2/atten_scores_2.sum() #전체합으로 나누어 합하면 1이 되도록 한다
print(atten_weights_2_tmp)
print(atten_weights_2_tmp.sum())
def softmax_navie(x):
  return torch.exp(x)/torch.exp(x).sum(dim=0) #소프트맥스= 항상 가중치가 양수되도록 보장(제곱이라서),기울기 반영
atten_weights_2=torch.softmax(atten_scores_2,dim=0)
print(atten_weights_2)#어텐션 가중치
print(atten_weights_2.sum())#가중치의 합

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
tensor(1.0000)
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [28]:
# ✏️ 책을 보며 직접 입력하세요: 컨텍스트 벡터 context_vec_2 (가중합)
context_vec_2=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
  context_vec_2+=atten_weights_2[i]*x_i #각 요소에 가중치를 곱해서 더한 합이 문맥 벡터가 된다(가중합)
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [29]:
# 🔍 확인
print("atten_scores_2:", atten_scores_2)
print("atten_weights_2:", atten_weights_2)
print("context_vec_2:", context_vec_2)
assert tuple(inputs.shape) == (6, 3), "inputs는 단어 6개 × 3차원이어야 해요"
assert _close(atten_scores_2, [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]), "어텐션 점수가 책과 달라요. 쿼리가 inputs[1]인지, 내적을 썼는지 확인"
assert _close(atten_weights_2.sum(), 1.0), "어텐션 가중치의 합은 1이어야 해요"
assert _close(context_vec_2, [0.4419, 0.6515, 0.5683]), "컨텍스트 벡터가 책과 달라요. torch.softmax로 정규화했는지 확인"
print("✅ 단어 하나의 컨텍스트 벡터 확인")

atten_scores_2: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
atten_weights_2: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
context_vec_2: tensor([0.4419, 0.6515, 0.5683])
✅ 단어 하나의 컨텍스트 벡터 확인


### 3.3.2 모든 입력 토큰에 대해 어텐션 가중치 계산하기
반복문 대신 **행렬 곱셈 `@`** 으로 6개 단어 전부를 한 번에 계산합니다. NER에서 배운 "반복문 대신 텐서 연산(벡터화)"과 같은 원리예요.

In [33]:
# ✏️ 책을 보며 직접 입력하세요: attn_scores = inputs @ inputs.T
atten_scores=torch.empty(6,6)
for i,x_i in enumerate(inputs):#각 x를 순회
  for j,x_j in enumerate(inputs): #각 x에 대해 닷연산한 결과 scores의 0번에 저장(0번에 0번 x의 가중치 저장)
    atten_scores[i,j]=torch.dot(x_i,x_j)
print(atten_scores)

atten_scores=inputs@inputs.T
print(atten_scores)



tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [37]:
# ✏️ 책을 보며 직접 입력하세요: attn_weights (행마다 softmax), all_context_vecs
atten_weights=torch.softmax(atten_scores,dim=-1)
print(atten_weights)
all_context_vecs=atten_weights@inputs
print(all_context_vecs)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [38]:
# 🔍 확인
print(all_context_vecs)
assert tuple(atten_scores.shape) == (6, 6), "단어 6개끼리의 점수라 (6, 6)이어야 해요"
assert _close(atten_weights.sum(dim=-1), [1.0] * 6), "각 행(쿼리 하나)의 가중치 합이 1이어야 해요. softmax의 dim=-1 확인"
assert _close(all_context_vecs[1], [0.4419, 0.6515, 0.5683]), "두 번째 행은 3.3.1의 context_vec_2와 같아야 해요"
print("✅ 행렬 곱셈으로 모든 컨텍스트 벡터 확인")

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
✅ 행렬 곱셈으로 모든 컨텍스트 벡터 확인


## 3.4 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기
### 3.4.1 단계별로 어텐션 가중치 계산하기
쿼리·키·밸류 가중치 행렬 `W_query`, `W_key`, `W_value`를 만들고, $\sqrt{d_k}$로 나누는 **스케일드 점곱 어텐션**을 계산합니다. 공부 노트의 $Q = XW_Q$, $K = XW_K$, $V = XW_V$예요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: d_in, d_out, torch.manual_seed(123), W_query / W_key / W_value

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: query_2, keys, values, attn_scores_2, attn_weights_2 (√d_k로 나누기), context_vec_2

In [ ]:
# 🔍 확인
print("query_2:", query_2)
print("context_vec_2:", context_vec_2)
assert tuple(keys.shape) == (6, 2) and tuple(values.shape) == (6, 2), "d_out=2라 keys, values는 (6, 2)여야 해요"
assert _close(query_2, [0.4306, 1.4551]), "query_2가 책과 달라요. W_query를 만들기 직전에 torch.manual_seed(123)을 했는지 확인"
assert _close(attn_weights_2, [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820]), "가중치가 책과 달라요. √d_k로 나눴는지 확인"
assert _close(context_vec_2, [0.3061, 0.8210]), "컨텍스트 벡터가 책과 달라요"
print("✅ 스케일드 점곱 어텐션 확인")

### 3.4.2 셀프 어텐션 파이썬 클래스 구현하기
3.4.1의 계산을 `nn.Module` 클래스로 묶습니다. v1은 `nn.Parameter`, v2는 `nn.Linear`를 써요. 2단계(NER)에서 쓴 `AutoModelForTokenClassification`도 이런 `nn.Module`이 쌓인 거예요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: import torch.nn as nn, SelfAttention_v1 클래스, sa_v1

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: SelfAttention_v2 클래스 (nn.Linear), sa_v2

In [ ]:
# 🔍 확인
torch.manual_seed(123)
_sa = SelfAttention_v1(3, 2)
print(_sa(inputs))
assert _close(_sa(inputs)[1], [0.3061, 0.8210]), "SelfAttention_v1의 두 번째 출력은 3.4.1의 context_vec_2와 같아야 해요"
assert tuple(sa_v2(inputs).shape) == (6, 2), "SelfAttention_v2 출력은 (6, 2)여야 해요"
print("✅ 셀프 어텐션 클래스 확인")

## 3.5 코잘 어텐션으로 미래의 단어를 감추기
### 3.5.1 코잘 어텐션 마스크 적용하기
GPT는 다음 단어를 맞혀야 하니 **뒤에 오는 단어를 보면 안 돼요.** 대각선 위쪽을 가리는 마스크를 만듭니다. 공부 노트의 "디코더 셀프 어텐션은 뒤 단어를 마스킹"이 이거예요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.tril 마스크로 곱하고 다시 정규화하는 방법

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.triu 마스크 + masked_fill(-torch.inf) + softmax → attn_weights

In [ ]:
# 🔍 확인: 대각선 위가 전부 0이고, 행마다 합이 1인지
print(attn_weights)
assert tuple(attn_weights.shape) == (6, 6)
assert torch.all(attn_weights.triu(diagonal=1) == 0), "대각선 위(미래 단어)의 가중치는 모두 0이어야 해요"
assert _close(attn_weights.sum(dim=-1), [1.0] * 6), "마스킹한 뒤에도 행마다 합이 1이어야 해요"
assert _close(attn_weights[1, :2], [0.5517, 0.4483]), "두 번째 행이 책과 달라요"
print("✅ 코잘 마스크 확인")

### 3.5.2 드롭아웃으로 어텐션 가중치에 추가 마스킹하기
학습할 때만 일부 가중치를 무작위로 0으로 만들어 과적합을 막습니다. 남은 값이 왜 커지는지(2배) 출력을 보며 확인하세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.nn.Dropout(0.5)를 예시 행렬과 attn_weights에 적용

### 3.5.3 코잘 어텐션 클래스 구현하기
배치 입력(`batch`: 같은 문장 2개)을 받는 `CausalAttention` 클래스를 만듭니다. `register_buffer`로 마스크를 모델에 붙이는 방법을 눈여겨보세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: batch = torch.stack((inputs, inputs), dim=0)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: CausalAttention 클래스, ca, context_vecs

In [ ]:
# 🔍 확인: 앞 단어의 출력은 뒤 단어가 바뀌어도 그대로인지 (코잘의 핵심)
_ca = CausalAttention(3, 2, 6, 0.0)
_out = _ca(batch)
_changed = batch.clone()
_changed[:, 3:, :] = 0.0  # 4번째 단어부터 바꿔 보기
_out2 = _ca(_changed)
print("context_vecs.shape:", context_vecs.shape)
assert tuple(context_vecs.shape) == (2, 6, 2), "(배치 2, 단어 6, d_out 2)여야 해요"
assert torch.allclose(_out[:, :3], _out2[:, :3]), "앞 3개 단어의 출력이 뒤 단어 때문에 바뀌었어요. 마스크를 확인하세요"
assert not torch.allclose(_out[:, 3:], _out2[:, 3:]), "뒤 단어를 바꿨으면 그 위치의 출력은 달라져야 해요"
print("✅ 코잘 어텐션: 미래 단어를 보지 않음")

## 3.6 싱글 헤드 어텐션을 멀티 헤드 어텐션으로 확장하기
### 3.6.1 여러 개의 싱글 헤드 어텐션 층 쌓기
`CausalAttention`을 여러 개 만들어 결과를 이어 붙이는 가장 단순한 방법이에요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: MultiHeadAttentionWrapper 클래스, mha, context_vecs

In [ ]:
# 🔍 확인
_mhw = MultiHeadAttentionWrapper(3, 2, 6, 0.0, num_heads=2)
print(_mhw(batch).shape)
assert tuple(_mhw(batch).shape) == (2, 6, 4), "헤드 2개 × d_out 2를 이어 붙여 마지막 차원이 4여야 해요"
print("✅ 멀티 헤드 래퍼 확인")

### 3.6.2 가중치 분할로 멀티 헤드 어텐션 구현하기 ⭐
3장의 최종 목표예요. 큰 가중치 행렬 하나로 계산한 뒤 `view`와 `transpose`로 헤드를 나눕니다. **텐서 모양이 줄마다 어떻게 바뀌는지** 주석으로 적어 가며 입력하세요. 4장의 GPT 모델에서 이 클래스를 그대로 씁니다.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: MultiHeadAttention 클래스 (view, transpose, contiguous)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2), context_vecs

In [ ]:
# 🔍 확인
_mha = MultiHeadAttention(3, 2, 6, 0.0, num_heads=2)
_out = _mha(batch)
print(_out.shape)
assert tuple(_out.shape) == (2, 6, 2), "가중치 분할 방식은 출력 차원이 d_out 그대로(2)여야 해요"
_changed = batch.clone()
_changed[:, 3:, :] = 0.0
assert torch.allclose(_out[:, :3], _mha(_changed)[:, :3]), "멀티 헤드에서도 앞 단어는 뒤 단어를 보면 안 돼요"
try:
    MultiHeadAttention(3, 3, 6, 0.0, num_heads=2)
    raise AssertionError("d_out이 num_heads로 나누어떨어지지 않으면 에러가 나야 해요")
except AssertionError as e:
    if "에러가 나야" in str(e):
        raise
print("✅ 멀티 헤드 어텐션 확인")

---
## 3장 마무리
- [ ] `notes.md` 채우기
- [ ] 이 노트북을 닫고 `CausalAttention`을 보지 않고 다시 짜기
- [ ] `MultiHeadAttention`을 보지 않고 다시 짜기 (텐서 모양을 주석으로 적으며). 다시 짠 코드는 `attention.py`로 저장해 4장에서 `import`해 보기
- [ ] 노트북을 .ipynb로 다운로드해 이 폴더에 넣고 커밋, push